In [ ]:

import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

# Step 1: Load a numerical dataset
iris = load_iris()
data = pd.DataFrame(iris.data, columns=iris.feature_names)

# Introduce a few missing values for demonstration
data.loc[0, "sepal length (cm)"] = None
data.loc[5, "petal length (cm)"] = None

print("Dataset shape:", data.shape)
print("\nMissing values before handling:")
print(data.isnull().sum())

# Step 2: Handle missing values
imputer = SimpleImputer(strategy="median")
X = imputer.fit_transform(data)

# Standardize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Step 3: Fit K-Means for different K values
inertia = []
silhouette_scores = []

from sklearn.metrics import silhouette_score

k_values = range(2, 7)

for k in k_values:
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = model.fit_predict(X_scaled)

    inertia.append(model.inertia_)
    silhouette_scores.append(silhouette_score(X_scaled, labels))

# Step 4: Plot the elbow method
plt.figure(figsize=(7, 4))
plt.plot(range(1, 7), [
    KMeans(n_clusters=k, random_state=42, n_init=10)
    .fit(X_scaled).inertia_
    for k in range(1, 7)
], marker="o")

plt.xlabel("Number of Clusters (K)")
plt.ylabel("Inertia")
plt.title("Elbow Method")
plt.grid(True)
plt.show()

# Display evaluation results
results = pd.DataFrame({
    "K": list(k_values),
    "Inertia": inertia,
    "Silhouette Score": silhouette_scores
})

print("\nEvaluation Results:")
print(results.round(3))

# Step 5: Select the final K
best_k = 3
final_model = KMeans(
    n_clusters=best_k, random_state=42, n_init=10
)
clusters = final_model.fit_predict(X_scaled)

data["Cluster"] = clusters

# Reduce features to 2D for visualization
pca = PCA(n_components=2)
X_2d = pca.fit_transform(X_scaled)

plt.figure(figsize=(8, 5))
scatter = plt.scatter(
    X_2d[:, 0], X_2d[:, 1],
    c=clusters, cmap="viridis", s=50
)

plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.title(f"K-Means Clusters (K={best_k})")
plt.colorbar(scatter, label="Cluster")
plt.grid(True)
plt.show()

# Interpret final clusters
print("\nNumber of samples in each cluster:")
print(data["Cluster"].value_counts().sort_index())

print("\nAverage feature values by cluster:")
print(data.groupby("Cluster").mean().round(2))
